# Business Entity Resolution Pipeline

Amazon ML Challenge 2026 — one notebook over the existing `code/business_entity_resolution/src/` modules.

**Task:** for each Source 1 entity, find zero, one, or many Source 2 / Source 3 matches. No external APIs or identity lookup.

**Reuse (do not copy-rewrite):**
- `src.preprocessing.loader` / `src.preprocessing.normalize`
- `src.blocking.*` via `src.evaluation.benchmark.get_blocker`
- `src.evaluation.metrics` (`blocking_metrics`, `f05_macro`, `f05_per_entity`)
- `student_resource/utils/validate_submission.py`

Pair features and the matcher live in this notebook because they are not in `src/` yet. Swap `BLOCKING_METHOD`, `FEATURE_NAMES`, or `MODEL` without rewriting later stages.

**Default is a sample run.** Set `RUN_FULL_INDEX = True` / `RUN_FULL_TEST = True` only when you want full S2/S3 indexes and Portal-ready test files. Cells print **measured** numbers only.


In [1]:
# === CONFIG (change these) ===
SAMPLE_SIZE = 10000          # S1 rows for inspect / train / val
BLOCKING_METHOD = "hybrid"   # exact_name | exact_address | token_name | hybrid
MODEL = "lightgbm"           # lightgbm (sklearn HistGB fallback if missing)
THRESHOLD = None             # None → Stage 8 picks the F0.5-maximizing cut

# Safety / cost (full challenge data is large — leave False until you mean it)
RUN_FULL_INDEX = False       # True: index ALL train S2/S3
INDEX_ROWS = 50_000          # S2 + S3 rows when RUN_FULL_INDEX is False
RUN_FULL_TEST = False        # True: full test inference + official output TSVs
VAL_FRACTION = 0.20
RANDOM_SEED = 42
MAX_NEGATIVES_PER_S1 = 20    # cap negative pairs per S1 for training


## STAGE 0 — SETUP

**What:** Resolve the repo root, put `src/` on `sys.path`, import existing modules, define helpers.

**Why:** Later stages must call the same loaders, normalizers, blockers, and F0.5 code as the rest of the repo.

**Input:** This notebook's location / current working directory.

**Output:** `ROOT`, imported APIs, `print_stage`, `need` / `mark` so you cannot skip stages.


In [2]:
from __future__ import annotations

import importlib.util
import sys
from pathlib import Path

import numpy as np
import pandas as pd


def find_root() -> Path:
    here = Path.cwd().resolve()
    for p in [here, *here.parents]:
        if (p / "student_resource" / "dataset").exists() and (
            p / "code" / "business_entity_resolution" / "src"
        ).exists():
            return p
    raise FileNotFoundError(
        "Could not find repo root (needs student_resource/dataset and "
        "code/business_entity_resolution/src). Open the notebook from the repo "
        "or start Jupyter at amazon-ml-challenge-2026/."
    )


ROOT = find_root()
SRC_PARENT = ROOT / "code" / "business_entity_resolution"
if str(SRC_PARENT) not in sys.path:
    sys.path.insert(0, str(SRC_PARENT))

from src.evaluation.benchmark import get_blocker  # noqa: E402
from src.evaluation.metrics import blocking_metrics, f05_macro, f05_per_entity  # noqa: E402
from src.preprocessing.loader import dataset_paths, load_ground_truth_sample, load_source  # noqa: E402
from src.preprocessing.normalize import (  # noqa: E402
    extract_numbers,
    normalize_address,
    normalize_country,
    normalize_name,
    tokenize,
)

STAGE_DONE: set[int] = set()
STAGE_NOTES: dict[int, dict] = {}


def mark(n: int, **note):
    STAGE_DONE.add(n)
    STAGE_NOTES[n] = note


def need(*ns: int):
    missing = [n for n in ns if n not in STAGE_DONE]
    if missing:
        raise RuntimeError(
            f"Run stage(s) {missing} first. Completed: {sorted(STAGE_DONE)}"
        )


def print_stage(name, inp, out, metrics, next_step):
    print("CURRENT STAGE:", name)
    print("INPUT:", inp)
    print("OUTPUT:", out)
    print("KEY METRICS:")
    if isinstance(metrics, dict):
        if not metrics:
            print("  (nothing measured yet — run the cells above)")
        for k, v in metrics.items():
            print(f"  {k}: {v}")
    else:
        print(" ", metrics)
    print("NEXT STEP:", next_step)


print("ROOT:", ROOT)
print("src on path:", SRC_PARENT)
print("BLOCKING_METHOD:", BLOCKING_METHOD, "| MODEL:", MODEL, "| SAMPLE_SIZE:", SAMPLE_SIZE)
print("RUN_FULL_INDEX:", RUN_FULL_INDEX, "| RUN_FULL_TEST:", RUN_FULL_TEST)
mark(0, root=str(ROOT))


ROOT: C:\Users\Admin\Desktop\AMAZON ML\amazon-ml-challenge-2026
src on path: C:\Users\Admin\Desktop\AMAZON ML\amazon-ml-challenge-2026\code\business_entity_resolution
BLOCKING_METHOD: hybrid | MODEL: lightgbm | SAMPLE_SIZE: 10000
RUN_FULL_INDEX: False | RUN_FULL_TEST: False


In [3]:
print_stage(
    name="STAGE 0 — SETUP",
    inp="notebook CWD + existing src/ tree",
    out="ROOT, get_blocker, loaders, normalize_*, F0.5 helpers",
    metrics={"repo_root": str(ROOT), "sample_size": SAMPLE_SIZE, "blocking": BLOCKING_METHOD},
    next_step="STAGE 1 — load train/test paths and a SAMPLE_SIZE slice (not the full 10M rows)",
)


CURRENT STAGE: STAGE 0 — SETUP
INPUT: notebook CWD + existing src/ tree
OUTPUT: ROOT, get_blocker, loaders, normalize_*, F0.5 helpers
KEY METRICS:
  repo_root: C:\Users\Admin\Desktop\AMAZON ML\amazon-ml-challenge-2026
  sample_size: 10000
  blocking: hybrid
NEXT STEP: STAGE 1 — load train/test paths and a SAMPLE_SIZE slice (not the full 10M rows)


## STAGE 1 — DATA LOADING

**What:** Locate official TSVs and load a **sample** of Source 1 plus a capped slice of Source 2 / Source 3. Ground truth is loaded only for the sampled S1 IDs.

**Why:** Inspect schema and label rates before paying for a full-file index.

**Input:** `student_resource/dataset/train|test/*.tsv` (tab-separated).

**Output:** `s1_train`, `s2_train`, `s3_train`, `truth_train` (dict), path map.

Columns: `entity_id`, `business_name`, `business_address`, `country`. GT: `source1_entity_id`, `matched_entity_ids`. Country is an open label set (test includes France).


In [5]:
need(0)

TRAIN_PATHS = dataset_paths(str(ROOT / "student_resource" / "dataset"), "train")
TEST_PATHS = dataset_paths(str(ROOT / "student_resource" / "dataset"), "test")


def load_tsv(path: str, nrows=None) -> pd.DataFrame:
    kwargs = dict(sep="\t", dtype=str, encoding="utf-8", na_filter=False)
    if nrows is not None:
        kwargs["nrows"] = nrows
    return pd.read_csv(path, **kwargs)


s2_nrows = None if RUN_FULL_INDEX else INDEX_ROWS
s3_nrows = None if RUN_FULL_INDEX else INDEX_ROWS

s1_train = load_tsv(TRAIN_PATHS["source1"], nrows=SAMPLE_SIZE)
s2_train = load_tsv(TRAIN_PATHS["source2"], nrows=s2_nrows)
s3_train = load_tsv(TRAIN_PATHS["source3"], nrows=s3_nrows)
truth_train = load_ground_truth_sample(TRAIN_PATHS["ground_truth"], set(s1_train["entity_id"]))

singleton_n = sum(1 for s1 in s1_train["entity_id"] if not truth_train.get(s1))
labeled_n = sum(1 for s1 in s1_train["entity_id"] if s1 in truth_train)

print(s1_train.head(3).to_string())
print("---")
print("S1 sample rows:", len(s1_train), "| S2 indexed rows:", len(s2_train), "| S3 indexed rows:", len(s3_train))
print("GT entries for this S1 sample:", labeled_n, "| singletons in sample:", singleton_n)
print("S1 columns:", list(s1_train.columns))
print("S1 countries (sample):", sorted(s1_train["country"].map(normalize_country).unique().tolist())[:20])
if not RUN_FULL_INDEX:
    print("NOTE: S2/S3 are capped at INDEX_ROWS. Blocking recall is measured on this reduced universe, not the full train sources.")

mark(1)


      entity_id        business_name                        business_address country
0  S1-925783039  Orelee's Barbershop  1795 Westchester Drive, High Point, NC      US
1  S1-773889195          Prime Money         17560 Ellis Road, Tahlequah, OK      US
2  S1-377745466        B+ Retail Inc     1712 Montebello Avenue, Phoenix, AZ      US
---
S1 sample rows: 10000 | S2 indexed rows: 50000 | S3 indexed rows: 50000
GT entries for this S1 sample: 10000 | singletons in sample: 560
S1 columns: ['entity_id', 'business_name', 'business_address', 'country']
S1 countries (sample): ['india', 'us']
NOTE: S2/S3 are capped at INDEX_ROWS. Blocking recall is measured on this reduced universe, not the full train sources.


In [6]:
need(1)
print_stage(
    name="STAGE 1 — DATA LOADING",
    inp=str(ROOT / "student_resource" / "dataset" / "train"),
    out="s1_train, s2_train, s3_train, truth_train",
    metrics={
        "s1_rows": len(s1_train),
        "s2_rows": len(s2_train),
        "s3_rows": len(s3_train),
        "gt_keys": len(truth_train),
        "singletons_in_sample": int(sum(1 for s1 in s1_train["entity_id"] if not truth_train.get(s1))),
        "full_s2s3_index": RUN_FULL_INDEX,
    },
    next_step="STAGE 2 — normalize names/addresses on the loaded sample",
)


CURRENT STAGE: STAGE 1 — DATA LOADING
INPUT: C:\Users\Admin\Desktop\AMAZON ML\amazon-ml-challenge-2026\student_resource\dataset\train
OUTPUT: s1_train, s2_train, s3_train, truth_train
KEY METRICS:
  s1_rows: 10000
  s2_rows: 50000
  s3_rows: 50000
  gt_keys: 10000
  singletons_in_sample: 560
  full_s2s3_index: False
NEXT STEP: STAGE 2 — normalize names/addresses on the loaded sample


## STAGE 2 — PREPROCESSING

**What:** Apply `normalize_name`, `normalize_address`, `normalize_country` from `src` (abbreviations, legal suffixes, unicode). Raw columns stay; derived columns are added.

**Why:** Blocking keys and pair features need consistent strings. Numbers are kept (street / PIN). Country is canonicalized but not restricted to {US, India}.

**Input:** Raw S1/S2/S3 sample frames.

**Output:** Same frames plus `name_norm`, `addr_norm`, `country_norm`.


In [7]:
need(1)


def add_normalized(df: pd.DataFrame) -> pd.DataFrame:
    out = df.copy()
    out["name_norm"] = out["business_name"].map(normalize_name)
    out["addr_norm"] = out["business_address"].map(normalize_address)
    out["country_norm"] = out["country"].map(normalize_country)
    return out


s1_train = add_normalized(s1_train)
s2_train = add_normalized(s2_train)
s3_train = add_normalized(s3_train)

preview = s1_train[["entity_id", "business_name", "name_norm", "business_address", "addr_norm", "country", "country_norm"]].head(5)
print(preview.to_string())
print("empty name_norm rate (S1):", float((s1_train["name_norm"] == "").mean()))
print("empty addr_norm rate (S1):", float((s1_train["addr_norm"] == "").mean()))

records_by_id = {}
for df in (s2_train, s3_train):
    for row in df.itertuples(index=False):
        d = row._asdict()
        records_by_id[d["entity_id"]] = d

mark(2, n_records=len(records_by_id))


      entity_id            business_name                name_norm                                      business_address                                             addr_norm country country_norm
0  S1-925783039      Orelee's Barbershop      orelee s barbershop                1795 Westchester Drive, High Point, NC      1795 westchester drive high point north carolina      US           us
1  S1-773889195              Prime Money              prime money                       17560 Ellis Road, Tahlequah, OK                   17560 ellis road tahlequah oklahoma      US           us
2  S1-377745466            B+ Retail Inc             b retail inc                   1712 Montebello Avenue, Phoenix, AZ                1712 montebello avenue phoenix arizona      US           us
3  S1-133037285            Christ Chapel            christ chapel     2100 Cameron Drive, Unit APARTMENT G, Dundalk, MD  2100 cameron drive unit apartment g dundalk maryland      US           us
4  S1-755362802  Prabhav 

In [8]:
need(2)
print_stage(
    name="STAGE 2 — PREPROCESSING",
    inp="raw business_name / business_address / country",
    out="name_norm, addr_norm, country_norm, records_by_id",
    metrics={
        "s2s3_lookup_size": len(records_by_id),
        "s1_empty_name_rate": float((s1_train["name_norm"] == "").mean()),
        "s1_empty_addr_rate": float((s1_train["addr_norm"] == "").mean()),
    },
    next_step="STAGE 3 — build BLOCKING_METHOD index on S2/S3 and retrieve candidates for sampled S1",
)


CURRENT STAGE: STAGE 2 — PREPROCESSING
INPUT: raw business_name / business_address / country
OUTPUT: name_norm, addr_norm, country_norm, records_by_id
KEY METRICS:
  s2s3_lookup_size: 100000
  s1_empty_name_rate: 0.0
  s1_empty_addr_rate: 0.0
NEXT STEP: STAGE 3 — build BLOCKING_METHOD index on S2/S3 and retrieve candidates for sampled S1


## STAGE 3 — BLOCKING / CANDIDATE GENERATION

**What:** `get_blocker(BLOCKING_METHOD)` from `src` indexes S2/S3 and returns a candidate ID set per S1 entity. Change `BLOCKING_METHOD` in the config cell only.

**Why:** Full S1×S2×S3 comparison does not scale. The candidate set is the **final** set the matcher will score (official `candidate_pairs.tsv` rule).

**Input:** Normalized S1 sample + indexed S2/S3 sample (or full if `RUN_FULL_INDEX`).

**Output:** `candidates_train: dict[s1_id, set[s2/s3 ids]]`.


In [9]:
need(2)

blocker = get_blocker(BLOCKING_METHOD)
print("Using blocker:", type(blocker).__name__, "method=", BLOCKING_METHOD)
blocker.build_index(s2_train, s3_train)

candidates_train = {}
for row in s1_train.itertuples(index=False):
    rec = row._asdict()
    candidates_train[rec["entity_id"]] = set(blocker.generate_candidates(rec))

n_cand = np.array([len(v) for v in candidates_train.values()])
print("S1 queried:", len(candidates_train))
print("avg candidates:", float(n_cand.mean()) if len(n_cand) else 0.0)
print("zero-candidate S1:", int((n_cand == 0).sum()) if len(n_cand) else 0)
print("example:", next(iter(candidates_train.items())))

mark(3)


Using blocker: HybridBlocker method= hybrid
S1 queried: 10000
avg candidates: 0.0958
zero-candidate S1: 9294
example: ('S1-925783039', set())


In [10]:
need(3)
n_cand = np.array([len(v) for v in candidates_train.values()])
print_stage(
    name="STAGE 3 — BLOCKING / CANDIDATE GENERATION",
    inp=f"S1 n={len(s1_train)}; S2 n={len(s2_train)}; S3 n={len(s3_train)}; method={BLOCKING_METHOD}",
    out="candidates_train",
    metrics={
        "method": BLOCKING_METHOD,
        "avg_candidates": float(n_cand.mean()) if len(n_cand) else 0.0,
        "median_candidates": float(np.median(n_cand)) if len(n_cand) else 0.0,
        "zero_candidate_rate": float((n_cand == 0).mean()) if len(n_cand) else 0.0,
        "index_is_full_train_s2s3": RUN_FULL_INDEX,
    },
    next_step="STAGE 4 — blocking_metrics vs train_ground_truth on this sample",
)


CURRENT STAGE: STAGE 3 — BLOCKING / CANDIDATE GENERATION
INPUT: S1 n=10000; S2 n=50000; S3 n=50000; method=hybrid
OUTPUT: candidates_train
KEY METRICS:
  method: hybrid
  avg_candidates: 0.0958
  median_candidates: 0.0
  zero_candidate_rate: 0.9294
  index_is_full_train_s2s3: False
NEXT STEP: STAGE 4 — blocking_metrics vs train_ground_truth on this sample


## STAGE 4 — BLOCKING EVALUATION

**What:** `blocking_metrics` from `src` — candidate recall (share of true S2/S3 links retrieved) and candidate-size stats.

**Why:** Recall of the matcher cannot exceed blocking recall. F0.5 on matches is a later stage.

**Input:** `candidates_train`, `truth_train`.

**Output:** Measured blocking stats (this sample / this index universe only).


In [11]:
need(3)

block_stats = blocking_metrics(candidates_train, truth_train)
for k, v in block_stats.items():
    print(f"{k}: {v}")

true_pairs = 0
found_pairs = 0
for s1, truth in truth_train.items():
    if not truth:
        continue
    true_pairs += len(truth)
    found_pairs += len(truth & candidates_train.get(s1, set()))
print("true_match_ids_in_gt:", true_pairs, "| of those also in candidates:", found_pairs)
print("true_match_ids_present_in_index:", sum(1 for s1, t in truth_train.items() for x in t if x in records_by_id))

mark(4, **{k: block_stats[k] for k in block_stats})


candidate_recall: 0.003481813996316759
avg_candidates: 0.0958
median_candidates: 0.0
p90_candidates: 0.0
p95_candidates: 1.0
p99_candidates: 2.0
zero_candidate_rate: 0.9294
true_match_ids_in_gt: 34752 | of those also in candidates: 121
true_match_ids_present_in_index: 320


In [12]:
need(4)
print_stage(
    name="STAGE 4 — BLOCKING EVALUATION",
    inp="candidates_train + truth_train",
    out="block_stats",
    metrics=block_stats,
    next_step="STAGE 5 — pairwise features on (S1, candidate) only — not the raw Cartesian product",
)


CURRENT STAGE: STAGE 4 — BLOCKING EVALUATION
INPUT: candidates_train + truth_train
OUTPUT: block_stats
KEY METRICS:
  candidate_recall: 0.003481813996316759
  avg_candidates: 0.0958
  median_candidates: 0.0
  p90_candidates: 0.0
  p95_candidates: 1.0
  p99_candidates: 2.0
  zero_candidate_rate: 0.9294
NEXT STEP: STAGE 5 — pairwise features on (S1, candidate) only — not the raw Cartesian product


## STAGE 5 — PAIR FEATURE ENGINEERING

**What:** Build labeled pairs from **candidates ∩ indexed records**. Features: RapidFuzz ratios, token Jaccard, address-number Jaccard, same-country. Edit `FEATURE_NAMES` / `pair_features` to swap the feature set.

**Why:** The matcher only sees pairs the blocker emitted. Positives = candidate IDs in ground truth; negatives = other candidates (capped per S1).

**Input:** `s1_train`, `candidates_train`, `records_by_id`, `truth_train`.

**Output:** `pairs_df` with `FEATURE_NAMES` + `label`.


In [ ]:
need(4)
from rapidfuzz.fuzz import ratio, token_set_ratio

FEATURE_NAMES = [
    "name_ratio",
    "name_token_set",
    "name_jaccard",
    "addr_ratio",
    "addr_token_set",
    "addr_jaccard",
    "num_jaccard",
    "same_country",
]


def _jaccard(a, b) -> float:
    sa, sb = set(a), set(b)
    if not sa and not sb:
        return 1.0
    if not sa or not sb:
        return 0.0
    return len(sa & sb) / len(sa | sb)


def pair_features(s1: dict, s2: dict) -> dict:
    n1, n2 = normalize_name(s1["business_name"]), normalize_name(s2["business_name"])
    a1, a2 = normalize_address(s1["business_address"]), normalize_address(s2["business_address"])
    c1, c2 = normalize_country(s1["country"]), normalize_country(s2["country"])
    return {
        "name_ratio": ratio(n1, n2) / 100.0,
        "name_token_set": token_set_ratio(n1, n2) / 100.0,
        "name_jaccard": _jaccard(tokenize(n1), tokenize(n2)),
        "addr_ratio": ratio(a1, a2) / 100.0,
        "addr_token_set": token_set_ratio(a1, a2) / 100.0,
        "addr_jaccard": _jaccard(tokenize(a1), tokenize(a2)),
        "num_jaccard": _jaccard(
            extract_numbers(s1["business_address"]),
            extract_numbers(s2["business_address"]),
        ),
        "same_country": float(bool(c1) and c1 == c2),
    }


rng = np.random.default_rng(RANDOM_SEED)
rows = []
skipped_missing = 0
for rec in s1_train.itertuples(index=False):
    s1 = rec._asdict()
    s1_id = s1["entity_id"]
    cands = list(candidates_train.get(s1_id, set()))
    truth = truth_train.get(s1_id, set())
    pos = [c for c in cands if c in truth]
    neg = [c for c in cands if c not in truth]
    if len(neg) > MAX_NEGATIVES_PER_S1:
        pick = rng.choice(len(neg), size=MAX_NEGATIVES_PER_S1, replace=False)
        neg = [neg[i] for i in pick]
    for cid in pos + neg:
        if cid not in records_by_id:
            skipped_missing += 1
            continue
        feat = pair_features(s1, records_by_id[cid])
        feat["s1_id"] = s1_id
        feat["cand_id"] = cid
        feat["label"] = int(cid in truth)
        rows.append(feat)

pairs_df = pd.DataFrame(rows)
print("pair rows:", len(pairs_df), "| skipped (cand not in index):", skipped_missing)
if len(pairs_df):
    print(pairs_df["label"].value_counts().to_string())
    print(pairs_df[FEATURE_NAMES + ["label"]].head(3).to_string())
else:
    print("No pairs. Increase INDEX_ROWS / SAMPLE_SIZE or set RUN_FULL_INDEX=True.")

mark(5, n_pairs=len(pairs_df))


In [ ]:
need(5)
metrics = {"n_pairs": int(len(pairs_df)), "feature_set": ",".join(FEATURE_NAMES)}
if len(pairs_df):
    metrics["n_positive"] = int(pairs_df["label"].sum())
    metrics["n_negative"] = int((pairs_df["label"] == 0).sum())
    metrics["positive_rate"] = float(pairs_df["label"].mean())
print_stage(
    name="STAGE 5 — PAIR FEATURE ENGINEERING",
    inp="candidate pairs only (not full S1×S2×S3)",
    out="pairs_df, FEATURE_NAMES, pair_features()",
    metrics=metrics,
    next_step="STAGE 6 — train/val split by S1 id, fit MODEL",
)


## STAGE 6 — MODEL TRAINING

**What:** Split **by Source 1 id** (not by pair) so validation entities are unseen. Fit LightGBM, or `HistGradientBoostingClassifier` if LightGBM is not installed. Swap `MODEL` in config.

**Why:** Supervised scoring of candidates. License constraint on the final model: MIT/Apache, ≤ 8B params (these models qualify).

**Input:** `pairs_df`.

**Output:** `model`, `s1_fit`, `s1_val`, train pair matrices.


In [ ]:
need(5)

if len(pairs_df) == 0:
    raise RuntimeError("No training pairs from Stage 5. Widen the sample/index before training.")

s1_ids = pairs_df["s1_id"].unique()
rng2 = np.random.default_rng(RANDOM_SEED)
perm = rng2.permutation(len(s1_ids))
s1_ids = s1_ids[perm]
n_val = max(1, int(len(s1_ids) * VAL_FRACTION))
s1_val = set(s1_ids[:n_val])
s1_fit = set(s1_ids[n_val:])
if not s1_fit:
    s1_fit, s1_val = s1_val, set()

fit_df = pairs_df[pairs_df["s1_id"].isin(s1_fit)]
val_pairs = pairs_df[pairs_df["s1_id"].isin(s1_val)] if s1_val else pairs_df.iloc[0:0]

X_fit = fit_df[FEATURE_NAMES].to_numpy(dtype=float)
y_fit = fit_df["label"].to_numpy(dtype=int)

model_name_used = MODEL
if MODEL == "lightgbm":
    try:
        import lightgbm as lgb

        model = lgb.LGBMClassifier(
            n_estimators=200,
            num_leaves=31,
            learning_rate=0.05,
            subsample=0.8,
            colsample_bytree=0.8,
            random_state=RANDOM_SEED,
            n_jobs=-1,
        )
    except ImportError:
        from sklearn.ensemble import HistGradientBoostingClassifier

        model = HistGradientBoostingClassifier(max_depth=6, learning_rate=0.05, random_state=RANDOM_SEED)
        model_name_used = "sklearn.HistGradientBoostingClassifier (lightgbm not installed)"
elif MODEL == "sklearn":
    from sklearn.ensemble import HistGradientBoostingClassifier

    model = HistGradientBoostingClassifier(max_depth=6, learning_rate=0.05, random_state=RANDOM_SEED)
    model_name_used = "sklearn.HistGradientBoostingClassifier"
else:
    raise ValueError(f"Unknown MODEL={MODEL!r}. Use 'lightgbm' or 'sklearn'.")

model.fit(X_fit, y_fit)
train_acc = float((model.predict(X_fit) == y_fit).mean())
print("model:", model_name_used)
print("fit S1:", len(s1_fit), "| val S1:", len(s1_val))
print("fit pairs:", len(fit_df), "pos:", int(y_fit.sum()), "neg:", int((y_fit == 0).sum()))
print("train pair accuracy (not F0.5):", train_acc)

mark(6, model=model_name_used, train_pair_acc=train_acc)


In [ ]:
need(6)
print_stage(
    name="STAGE 6 — MODEL TRAINING",
    inp=f"pairs_df split by S1 (val_fraction={VAL_FRACTION})",
    out="model, s1_fit, s1_val, fit_df, val_pairs",
    metrics={
        "model": model_name_used,
        "n_fit_s1": len(s1_fit),
        "n_val_s1": len(s1_val),
        "n_fit_pairs": int(len(fit_df)),
        "train_pair_accuracy": float((model.predict(X_fit) == y_fit).mean()),
    },
    next_step="STAGE 7 — entity-level macro F0.5 on validation S1 (not pair accuracy)",
)


## STAGE 7 — VALIDATION + F0.5

**What:** Score every candidate for each validation S1, apply `THRESHOLD` or 0.5, compute **macro F0.5** with `f05_macro` (singletons included; empty vs empty is 1.0; false merge on a singleton is 0.0).

**Why:** Leaderboard metric is entity-level F0.5, precision-weighted (β=0.5), not pair accuracy.

**Input:** `model`, validation S1 rows, `candidates_train`, `truth_train`.

**Output:** `val_pred`, `val_f05` at the working threshold.


In [ ]:
need(6)


def predict_entities(s1_df, cand_map, recs, clf, threshold: float) -> dict[str, set[str]]:
    out = {}
    for rec in s1_df.itertuples(index=False):
        s1 = rec._asdict()
        sid = s1["entity_id"]
        kept = set()
        for cid in cand_map.get(sid, set()):
            if cid not in recs:
                continue
            x = np.array([pair_features(s1, recs[cid])[k] for k in FEATURE_NAMES], dtype=float).reshape(1, -1)
            proba = float(clf.predict_proba(x)[0, 1])
            if proba >= threshold:
                kept.add(cid)
        out[sid] = kept
    return out


working_threshold = 0.5 if THRESHOLD is None else float(THRESHOLD)
s1_val_df = s1_train[s1_train["entity_id"].isin(s1_val)] if s1_val else s1_train.iloc[0:0]

if len(s1_val_df) == 0:
    print("Validation S1 set is empty (too few pair-generating entities). Using fit S1 as a diagnostic only.")
    s1_val_df = s1_train[s1_train["entity_id"].isin(s1_fit)]

val_pred = predict_entities(s1_val_df, candidates_train, records_by_id, model, working_threshold)
val_truth = {k: truth_train.get(k, set()) for k in val_pred}
val_f05 = float(f05_macro(val_pred, val_truth))

n_empty_pred = sum(1 for v in val_pred.values() if not v)
n_singleton_truth = sum(1 for k in val_pred if not val_truth[k])
print("working_threshold:", working_threshold)
print("val S1:", len(val_pred), "| macro F0.5:", val_f05)
print("empty predictions:", n_empty_pred, "| singleton GT entities:", n_singleton_truth)
print("example entity F0.5:", f05_per_entity(next(iter(val_pred.values())), val_truth[next(iter(val_pred))]))

mark(7, val_f05=val_f05, working_threshold=working_threshold)


In [ ]:
need(7)
print_stage(
    name="STAGE 7 — VALIDATION + F0.5",
    inp=f"val S1 + candidates + model @ threshold={working_threshold}",
    out="val_pred, val_f05",
    metrics={
        "n_val_s1": len(val_pred),
        "macro_f05": val_f05,
        "threshold": working_threshold,
        "empty_predictions": int(sum(1 for v in val_pred.values() if not v)),
    },
    next_step="STAGE 8 — sweep thresholds; pick the cut that maximizes macro F0.5",
)


## STAGE 8 — THRESHOLD OPTIMIZATION

**What:** Sweep probability cuts on the **same** validation candidate sets. Keep the threshold with highest macro F0.5. If config `THRESHOLD` is set, that value is used and the sweep is reported only.

**Why:** F0.5 punishes false merges more than misses; the default 0.5 is often too aggressive or too timid.

**Input:** `model`, `s1_val_df`, candidates, GT.

**Output:** `BEST_THRESHOLD`, `sweep_df`.


In [ ]:
need(7)

cuts = np.round(np.linspace(0.10, 0.90, 17), 2)
rows_t = []
for t in cuts:
    pred_t = predict_entities(s1_val_df, candidates_train, records_by_id, model, float(t))
    truth_t = {k: truth_train.get(k, set()) for k in pred_t}
    score = float(f05_macro(pred_t, truth_t))
    rows_t.append(
        {
            "threshold": float(t),
            "macro_f05": score,
            "mean_matches": float(np.mean([len(v) for v in pred_t.values()])),
        }
    )

sweep_df = pd.DataFrame(rows_t)
print(sweep_df.to_string(index=False))
best_row = sweep_df.loc[sweep_df["macro_f05"].idxmax()]
swept_best = float(best_row["threshold"])
BEST_THRESHOLD = float(THRESHOLD) if THRESHOLD is not None else swept_best
print("sweep_argmax_threshold:", swept_best, "F0.5:", float(best_row["macro_f05"]))
print("BEST_THRESHOLD (used going forward):", BEST_THRESHOLD, "(config THRESHOLD override)" if THRESHOLD is not None else "(from sweep)")

mark(8, BEST_THRESHOLD=BEST_THRESHOLD, sweep_best_f05=float(best_row["macro_f05"]))


In [ ]:
need(8)
print_stage(
    name="STAGE 8 — THRESHOLD OPTIMIZATION",
    inp="validation candidate scores",
    out="BEST_THRESHOLD, sweep_df",
    metrics={
        "BEST_THRESHOLD": BEST_THRESHOLD,
        "sweep_best_macro_f05": float(sweep_df["macro_f05"].max()),
        "config_THRESHOLD": THRESHOLD,
    },
    next_step="STAGE 9 — refit on all sample pairs (still SAMPLE_SIZE unless you change config)",
)


## STAGE 9 — FINAL TRAINING

**What:** Refit the same model class on **all sample pairs** (fit + val). This is still the SAMPLE_SIZE universe unless you raise `SAMPLE_SIZE` / `RUN_FULL_INDEX`.

**Why:** Use every labeled candidate pair before test inference.

**Input:** `pairs_df`, `FEATURE_NAMES`, `MODEL`.

**Output:** `final_model`.


In [ ]:
need(8)

X_all = pairs_df[FEATURE_NAMES].to_numpy(dtype=float)
y_all = pairs_df["label"].to_numpy(dtype=int)

if MODEL == "lightgbm":
    try:
        import lightgbm as lgb

        final_model = lgb.LGBMClassifier(
            n_estimators=200,
            num_leaves=31,
            learning_rate=0.05,
            subsample=0.8,
            colsample_bytree=0.8,
            random_state=RANDOM_SEED,
            n_jobs=-1,
        )
    except ImportError:
        from sklearn.ensemble import HistGradientBoostingClassifier

        final_model = HistGradientBoostingClassifier(max_depth=6, learning_rate=0.05, random_state=RANDOM_SEED)
else:
    from sklearn.ensemble import HistGradientBoostingClassifier

    final_model = HistGradientBoostingClassifier(max_depth=6, learning_rate=0.05, random_state=RANDOM_SEED)

final_model.fit(X_all, y_all)
print("final_model fitted on", len(pairs_df), "pairs; pos", int(y_all.sum()))
print("BEST_THRESHOLD:", BEST_THRESHOLD)

mark(9)


In [ ]:
need(9)
print_stage(
    name="STAGE 9 — FINAL TRAINING",
    inp="all sample pairs_df",
    out="final_model",
    metrics={"n_pairs": int(len(pairs_df)), "n_positive": int(pairs_df["label"].sum()), "threshold": BEST_THRESHOLD},
    next_step="STAGE 10 — test inference (SAMPLE_SIZE test S1 unless RUN_FULL_TEST=True)",
)


## STAGE 10 — TEST INFERENCE

**What:** Load test TSVs (sample or full), rebuild the **same** `BLOCKING_METHOD` blocker on test S2/S3, score candidates with `final_model` ≥ `BEST_THRESHOLD`.

**Why:** Test has no ground truth. Every test S1 must appear in the Portal file; France must not be dropped.

**Input:** `student_resource/dataset/test`, `final_model`, `BEST_THRESHOLD`.

**Output:** `candidates_test`, `pred_test`.

If `RUN_FULL_TEST` is False, only `SAMPLE_SIZE` test S1 rows and `INDEX_ROWS` S2/S3 are used — **not** a valid leaderboard submission.


In [ ]:
need(9)

test_s1_n = None if RUN_FULL_TEST else SAMPLE_SIZE
test_s23_n = None if RUN_FULL_TEST else INDEX_ROWS

s1_test = load_tsv(TEST_PATHS["source1"], nrows=test_s1_n)
s2_test = load_tsv(TEST_PATHS["source2"], nrows=test_s23_n)
s3_test = load_tsv(TEST_PATHS["source3"], nrows=test_s23_n)
s1_test = add_normalized(s1_test)
s2_test = add_normalized(s2_test)
s3_test = add_normalized(s3_test)

test_records = {}
for df in (s2_test, s3_test):
    for row in df.itertuples(index=False):
        d = row._asdict()
        test_records[d["entity_id"]] = d

test_blocker = get_blocker(BLOCKING_METHOD)
test_blocker.build_index(s2_test, s3_test)

candidates_test = {}
for row in s1_test.itertuples(index=False):
    rec = row._asdict()
    candidates_test[rec["entity_id"]] = set(test_blocker.generate_candidates(rec))

pred_test = predict_entities(s1_test, candidates_test, test_records, final_model, BEST_THRESHOLD)

tc = np.array([len(candidates_test[i]) for i in s1_test["entity_id"]])
tm = np.array([len(pred_test[i]) for i in s1_test["entity_id"]])
print("RUN_FULL_TEST:", RUN_FULL_TEST)
print("test S1 rows:", len(s1_test), "| countries:", sorted(s1_test["country_norm"].unique().tolist()))
print("avg candidates:", float(tc.mean()), "| avg predicted matches:", float(tm.mean()))
print("S1 with zero candidates:", int((tc == 0).sum()), "| S1 with zero matches:", int((tm == 0).sum()))

mark(10)


In [ ]:
need(10)
tc = np.array([len(candidates_test[i]) for i in s1_test["entity_id"]])
tm = np.array([len(pred_test[i]) for i in s1_test["entity_id"]])
print_stage(
    name="STAGE 10 — TEST INFERENCE",
    inp="test sources + final_model + BEST_THRESHOLD",
    out="candidates_test, pred_test, s1_test",
    metrics={
        "run_full_test": RUN_FULL_TEST,
        "n_s1": len(s1_test),
        "avg_candidates": float(tc.mean()),
        "avg_matches": float(tm.mean()),
        "zero_match_rate": float((tm == 0).mean()),
    },
    next_step="STAGE 11 — write tab-separated matching_results.tsv and candidate_pairs.tsv",
)


## STAGE 11 — OUTPUT GENERATION

**What:** Write official TSV schema: one row per S1, comma-separated S2/S3 IDs, empty string for singletons, matches ⊆ candidates.

**Why:** Portal scores `matching_results.tsv` only; the zip also needs `candidate_pairs.tsv` as the last candidate list the model scored.

**Input:** `s1_test`, `pred_test`, `candidates_test`.

**Output:** files under `output/` (sample prefix unless `RUN_FULL_TEST`).


In [ ]:
need(10)

out_dir = ROOT / "output"
out_dir.mkdir(exist_ok=True)
prefix = "" if RUN_FULL_TEST else "sample_"
match_path = out_dir / f"{prefix}matching_results.tsv"
cand_path = out_dir / f"{prefix}candidate_pairs.tsv"


def pack_ids(ids: set[str]) -> str:
    return ",".join(sorted(ids))


match_rows = [
    {"source1_entity_id": sid, "matched_entity_ids": pack_ids(pred_test[sid])}
    for sid in s1_test["entity_id"]
]
cand_rows = [
    {"source1_entity_id": sid, "candidate_entity_ids": pack_ids(candidates_test[sid])}
    for sid in s1_test["entity_id"]
]
match_df = pd.DataFrame(match_rows)
cand_df = pd.DataFrame(cand_rows)
match_df.to_csv(match_path, sep="\t", index=False, encoding="utf-8")
cand_df.to_csv(cand_path, sep="\t", index=False, encoding="utf-8")

subset_ok = True
for sid in s1_test["entity_id"]:
    if pred_test[sid] - candidates_test[sid]:
        subset_ok = False
        break

print("wrote", match_path)
print("wrote", cand_path)
print(match_df.head(3).to_string())
print("rows:", len(match_df), "| matches ⊆ candidates:", subset_ok)
if not RUN_FULL_TEST:
    print("SAMPLE output only — do not upload to the Portal until RUN_FULL_TEST=True and every test S1 has a row.")

mark(11, match_path=str(match_path), cand_path=str(cand_path))


In [ ]:
need(11)
print_stage(
    name="STAGE 11 — OUTPUT GENERATION",
    inp="pred_test + candidates_test",
    out=str(match_path) + " ; " + str(cand_path),
    metrics={
        "rows": int(len(match_df)),
        "portal_ready": bool(RUN_FULL_TEST),
        "matches_subset_of_candidates": bool(
            all(pred_test[s] <= candidates_test[s] for s in s1_test["entity_id"])
        ),
    },
    next_step="STAGE 12 — official validate_submission.py (full test files only)",
)


## STAGE 12 — OFFICIAL VALIDATION

**What:** Run `student_resource/utils/validate_submission.py` checks (headers, one row per test S1, S2/S3 prefixes, no duplicate IDs, matches ⊆ candidates).

**Why:** Catch format rejection before spending a Portal submission. This does **not** compute F0.5.

**Input:** output TSVs + `dataset/test`.

**Output:** PASS/FAIL text from the official validator.

Skipped automatically when `RUN_FULL_TEST` is False, because a sample file is missing S1 entities and would FAIL by design.


In [ ]:
need(11)

spec = importlib.util.spec_from_file_location(
    "validate_submission",
    ROOT / "student_resource" / "utils" / "validate_submission.py",
)
validate_mod = importlib.util.module_from_spec(spec)
spec.loader.exec_module(validate_mod)

if not RUN_FULL_TEST:
    print("SKIPPED official validator: RUN_FULL_TEST is False.")
    print("Sample TSVs do not contain every test S1 row, so validation would FAIL on missing entities.")
    print("Re-run Stages 10–12 with RUN_FULL_TEST=True to score format for the Portal.")
    errors, warnings = ["skipped: sample mode"], []
else:
    errors, warnings = validate_mod.validate(
        str(match_path),
        str(cand_path),
        str(ROOT / "student_resource" / "dataset" / "test"),
        check_ids=False,
    )
    for w in warnings:
        print("WARNING:", w)
    if errors:
        print("FAIL:")
        for i, e in enumerate(errors, 1):
            print(f"  {i}. {e}")
    else:
        print("PASS — format is safe to submit (F0.5 is computed on the Portal, not here).")

mark(12)


In [ ]:
need(12)
print_stage(
    name="STAGE 12 — OFFICIAL VALIDATION",
    inp=f"{match_path} + test_source1.tsv",
    out="validator errors/warnings",
    metrics={
        "run_full_test": RUN_FULL_TEST,
        "n_errors": 0 if errors == ["skipped: sample mode"] else len(errors),
        "status": (
            "skipped_sample_mode"
            if errors == ["skipped: sample mode"]
            else ("PASS" if not errors else "FAIL")
        ),
    },
    next_step="If sample looks sane: set RUN_FULL_INDEX=True, raise SAMPLE_SIZE, then RUN_FULL_TEST=True and upload output/matching_results.tsv",
)
